# Start of Month Rebalance: 70% QQQ, 20% BIL, 10% GLD

This notebook runs a simple **start-of-month** rebalance strategy using TiPortfolio with data from **Alpaca** (optional cache via helpers). Set `APCA_API_KEY_ID` and `APCA_API_SECRET_KEY` in your environment or `.env` before running.

In [1]:
from tiportfolio.helpers.cache import enable_data_source_cache
from tiportfolio import ScheduleBasedEngine, FixRatio, Schedule
from tiportfolio.data import prices_dict_from_long_format
from tiportfolio.helpers.data import Alpaca

from dotenv import load_dotenv
import os
load_dotenv()

alpaca = Alpaca(os.environ['ALPACA_API_KEY'], os.environ['ALPACA_API_SECRET'])

# Optional: enable disk cache so repeated runs use cached data
# from tiportfolio.helpers.cache import enable_data_source_cache
enable_data_source_cache("tiportfolio", cache_dir=".cache")

SYMBOLS = ["QQQ", "BIL", "GLD"]
WEIGHTS = {"QQQ": 0.7, "BIL": 0.2, "GLD": 0.1}
START = "2018-01-01"
END = "2024-12-31"
INITIAL_VALUE = 10_000  # starting portfolio value

In [2]:

df = alpaca.query(SYMBOLS, START, END, timeframe="1d", adjust="all")
prices = prices_dict_from_long_format(df)
print(f"Loaded {len(prices)} symbols, {sum(len(v) for v in prices.values())} total rows")

Loaded cached bar data.

Loaded 3 symbols, 5280 total rows


In [3]:
engine = ScheduleBasedEngine(
    allocation=FixRatio(weights=WEIGHTS),
    rebalance=Schedule("month_start"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result = engine.run(symbols=SYMBOLS, start=START, end=END, prices_df=prices)
print(result.summary())

Backtest Summary
----------------
Sharpe Ratio:    0.6930
CAGR:            15.36%
Max Drawdown:    26.33%
MAR Ratio:       0.5835
Kelly Leverage:  4.0825
Final Value:      27,161.39
Total Fee:        0.78
Rebalances:      83


In [4]:
from tiportfolio.report import plot_portfolio_with_assets_interactive

fig = plot_portfolio_with_assets_interactive(result, mark_rebalances=True)
fig.show()

In [5]:
from tiportfolio.report import rebalance_decisions_table

decisions_df = rebalance_decisions_table(result)
decisions_df.head(10)

,date,equity_before,equity_after,fee_paid,QQQ_price,QQQ_qty_before,QQQ_trade_qty,QQQ_qty_after,QQQ_value_after,BIL_price,BIL_qty_before,BIL_trade_qty,BIL_qty_after,BIL_value_after,GLD_price,GLD_qty_before,GLD_trade_qty,GLD_qty_after,GLD_value_after
0,2018-02-01 05:00:00+00:00,10444.058,10444.051,0.007,158.89,46.688,-0.676,46.012,7310.841,75.17,26.638,1.150,27.788,2088.812,128.07,7.990,0.165,8.155,1044.406
1,2018-03-01 05:00:00+00:00,10267.305,10267.302,0.003,155.60,46.012,0.178,46.190,7187.113,75.24,27.788,-0.496,27.292,2053.461,124.72,8.155,0.077,8.232,1026.730
2,2018-04-02 04:00:00+00:00,9910.068,9910.060,0.008,147.36,46.190,0.886,47.076,6937.048,75.33,27.292,-0.981,26.311,1982.014,127.26,8.232,-0.445,7.787,991.007
3,2018-05-02 04:00:00+00:00,10165.836,10165.829,0.006,153.34,47.076,-0.668,46.407,7116.085,75.42,26.311,0.647,26.958,2033.167,123.65,7.787,0.434,8.221,1016.584
4,2018-06-04 04:00:00+00:00,10707.245,10707.235,0.010,165.16,46.407,-1.027,45.381,7495.072,75.54,26.958,1.391,28.349,2141.449,122.39,8.221,0.527,8.748,1070.725
5,2018-07-02 04:00:00+00:00,10618.109,10618.107,0.002,164.09,45.381,-0.084,45.296,7432.676,75.63,28.349,-0.269,28.079,2123.622,117.46,8.748,0.291,9.040,1061.811
6,2018-08-02 04:00:00+00:00,10884.617,10884.611,0.006,170.49,45.296,-0.606,44.690,7619.232,75.75,28.079,0.659,28.738,2176.923,114.50,9.040,0.466,9.506,1088.462
7,2018-09-04 04:00:00+00:00,11140.836,11140.830,0.005,176.48,44.690,-0.501,44.190,7798.585,75.87,28.738,0.630,29.368,2228.167,112.93,9.506,0.359,9.865,1114.084
8,2018-10-02 04:00:00+00:00,11163.503,11163.503,0.000,176.71,44.190,0.032,44.222,7814.452,75.98,29.368,0.017,29.385,2232.701,113.87,9.865,-0.062,9.804,1116.350
9,2018-11-02 04:00:00+00:00,10505.600,10505.586,0.014,161.13,44.222,1.418,45.640,7353.920,76.11,29.385,-1.779,27.606,2101.120,116.65,9.804,-0.798,9.006,1050.560


In [6]:
decisions_df.tail(10)

,date,equity_before,equity_after,fee_paid,QQQ_price,QQQ_qty_before,QQQ_trade_qty,QQQ_qty_after,QQQ_value_after,BIL_price,BIL_qty_before,BIL_trade_qty,BIL_qty_after,BIL_value_after,GLD_price,GLD_qty_before,GLD_trade_qty,GLD_qty_after,GLD_value_after
73,2024-03-01 05:00:00+00:00,23612.537,23612.527,0.010,440.77,38.030,-0.530,37.500,16528.776,83.91,54.264,2.017,56.281,4722.507,192.89,11.907,0.335,12.241,2361.254
74,2024-04-02 04:00:00+00:00,23707.645,23707.639,0.005,436.89,37.500,0.485,37.985,16595.351,84.27,56.281,-0.015,56.266,4741.529,210.89,12.241,-1.000,11.242,2370.764
75,2024-05-02 04:00:00+00:00,23220.318,23220.310,0.008,422.82,37.985,0.457,38.442,16254.223,84.66,56.266,-1.410,54.855,4644.064,213.13,11.242,-0.347,10.895,2322.032
76,2024-06-03 04:00:00+00:00,24282.263,24282.252,0.011,448.80,38.442,-0.569,37.873,16997.584,85.00,54.855,2.279,57.135,4856.453,217.22,10.895,0.284,11.179,2428.226
77,2024-07-02 04:00:00+00:00,25583.333,25583.318,0.015,483.10,37.873,-0.804,37.070,17908.333,85.36,57.135,2.807,59.942,5116.667,215.56,11.179,0.690,11.868,2558.333
78,2024-08-02 04:00:00+00:00,24318.301,24318.282,0.019,445.18,37.070,1.168,38.238,17022.811,85.77,59.942,-3.236,56.706,4863.660,225.34,11.868,-1.076,10.792,2431.830
79,2024-09-03 04:00:00+00:00,24887.885,24887.880,0.005,458.13,38.238,-0.211,38.027,17421.519,86.14,56.706,1.079,57.785,4977.577,230.29,10.792,0.015,10.807,2488.788
80,2024-10-02 04:00:00+00:00,25859.483,25859.474,0.009,478.78,38.027,-0.220,37.808,18101.638,86.49,57.785,2.013,59.798,5171.897,245.66,10.807,-0.281,10.527,2585.948
81,2024-11-04 05:00:00+00:00,26109.449,26109.447,0.002,482.81,37.808,0.047,37.855,18276.614,86.86,59.798,0.321,60.118,5221.890,252.83,10.527,-0.200,10.327,2610.945
82,2024-12-02 05:00:00+00:00,27131.106,27131.093,0.013,511.90,37.855,-0.754,37.101,18991.774,87.15,60.118,2.145,62.263,5426.221,243.44,10.327,0.818,11.145,2713.111


## Comparison to Long QQQ

In [7]:
# Strategy 2: 100% QQQ (buy-and-hold, no rebalancing)
prices_qqq = {"QQQ": prices["QQQ"]}
engine_qqq = ScheduleBasedEngine(
    allocation=FixRatio(weights={"QQQ": 1.0}),
    rebalance=Schedule("month_start"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_qqq_only = engine_qqq.run(symbols=["QQQ"], start=START, end=END, prices_df=prices_qqq)
print("100% QQQ Summary")
print(result_qqq_only.summary())

100% QQQ Summary
Backtest Summary
----------------
Sharpe Ratio:    0.6860
CAGR:            19.24%
Max Drawdown:    35.01%
MAR Ratio:       0.5495
Kelly Leverage:  2.8433
Final Value:      34,218.64
Total Fee:        0.00
Rebalances:      0


In [12]:
# Compare: 70/20/10 rebalance vs 100% QQQ (verify which is better)
from tiportfolio import compare_strategies

comparison = compare_strategies(
    result,
    result_qqq_only,
    names=['rebalance', 'long qqq']
)
comparison

,rebalance,long qqq,better
metric,,,
sharpe_ratio,0.692978,0.685981,rebalance
cagr,0.153612,0.192355,long qqq
max_drawdown,0.263272,0.350069,rebalance
mar_ratio,0.583474,0.549477,rebalance
final_value,27161.385311,34218.635363,long qqq
kelly_leverage,4.082508,2.843300,rebalance
total_fee,0.784122,0.000000,long qqq


In [10]:
# Strategy 3: Never rebalance 70/20/10 QQQ/BIL/GLD
engine_never = ScheduleBasedEngine(
    allocation=FixRatio(weights=WEIGHTS),
    rebalance=Schedule("never"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_never = engine_never.run(symbols=SYMBOLS, start=START, end=END, prices_df=prices)
print("Never Rebalance 70/20/10 Summary")
print(result_never.summary())

Never Rebalance 70/20/10 Summary
Backtest Summary
----------------
Sharpe Ratio:    0.6710
CAGR:            15.99%
Max Drawdown:    30.14%
MAR Ratio:       0.5304
Kelly Leverage:  3.5626
Final Value:      28,205.56
Total Fee:        0.00
Rebalances:      0


In [11]:
## Compare All Strategies

compare_strategies(result, result_qqq_only, result_never)

,Strategy 1,Strategy 2,Strategy 3,better
metric,,,,
sharpe_ratio,0.692978,0.685981,0.670993,Strategy 1
cagr,0.153612,0.192355,0.159853,Strategy 2
max_drawdown,0.263272,0.350069,0.301358,Strategy 1
mar_ratio,0.583474,0.549477,0.530441,Strategy 1
final_value,27161.385311,34218.635363,28205.559041,Strategy 2
kelly_leverage,4.082508,2.843300,3.562556,Strategy 1
total_fee,0.784122,0.000000,0.000000,Strategy 2
